# Comparing Baseline and Hyperbolic Memory Models

This notebook compares the performance of:
1. **MemoryMLP** (Baseline): Standard Euclidean MLP from the TTT paper
2. **HyperbolicMemoryMLP**: MLP operating in hyperbolic (Poincaré ball) space

We'll train both models on a sequence memorization task to evaluate their ability to store and retrieve information.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import time
from collections import defaultdict

# Import memory models
from titans_pytorch.memory_models import MemoryMLP, HyperbolicMemoryMLP

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Create Synthetic Dataset

We'll create a sequence memorization task where the model needs to:
1. Read a sequence of vectors
2. Memorize patterns in the sequence
3. Predict the next vector in the sequence

This tests the model's ability to store and retrieve information.

In [ ]:
class SequenceMemorizationDataset(Dataset):
    """
    Dataset for sequence memorization task.
    
    Each sequence consists of:
    - Input: sequence of random vectors
    - Target: next vector in the sequence (shifted by 1)
    
    The task requires the model to memorize patterns and predict continuations.
    """
    
    def __init__(self, num_sequences=1000, seq_length=32, dim=64):
        """
        Args:
            num_sequences: Number of training sequences
            seq_length: Length of each sequence
            dim: Dimensionality of each vector
        """
        self.num_sequences = num_sequences
        self.seq_length = seq_length
        self.dim = dim
        
        # Pre-generate all sequences for consistency
        self.sequences = torch.randn(num_sequences, seq_length + 1, dim)
    
    def __len__(self):
        return self.num_sequences
    
    def __getitem__(self, idx):
        # Input: all vectors except the last
        # Target: all vectors except the first (shifted by 1)
        seq = self.sequences[idx]
        x = seq[:-1]  # Input sequence
        y = seq[1:]   # Target sequence (next step prediction)
        return x, y

# Create datasets
dim = 64
seq_length = 32

train_dataset = SequenceMemorizationDataset(num_sequences=1000, seq_length=seq_length, dim=dim)
val_dataset = SequenceMemorizationDataset(num_sequences=200, seq_length=seq_length, dim=dim)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training sequences: {len(train_dataset)}")
print(f"Validation sequences: {len(val_dataset)}")
print(f"Sequence length: {seq_length}")
print(f"Vector dimension: {dim}")

## 3. Define Model Wrapper

We'll wrap the memory models in a simple architecture that:
1. Processes the sequence through the memory model
2. Adds a final projection layer for predictions

In [ ]:
class MemoryModelWrapper(nn.Module):
    """
    Wrapper for memory models that adds a final projection.
    
    Architecture:
    1. Input sequence -> Memory Model (stores and retrieves)
    2. Memory output -> Prediction (next vector)
    """
    
    def __init__(self, memory_model, dim):
        """
        Args:
            memory_model: Either MemoryMLP or HyperbolicMemoryMLP
            dim: Vector dimension
        """
        super().__init__()
        self.memory = memory_model
        # No projection needed - memory model already outputs correct dimension
    
    def forward(self, x):
        """
        Args:
            x: Input sequence (batch, seq_len, dim)
        Returns:
            Predictions for next vectors (batch, seq_len, dim)
        """
        # Pass through memory model
        # Memory model transforms each position based on context
        out = self.memory(x)
        return out

# Create baseline model (Euclidean MLP)
baseline_memory = MemoryMLP(
    dim=dim,
    depth=2,
    expansion_factor=4.0
)
baseline_model = MemoryModelWrapper(baseline_memory, dim).to(device)

# Create hyperbolic model (Poincaré ball MLP)
hyperbolic_memory = HyperbolicMemoryMLP(
    dim=dim,
    depth=2,
    expansion_factor=4.0,
    curvature=1.0,
    learn_curvature=False  # Fixed curvature for this experiment
)
hyperbolic_model = MemoryModelWrapper(hyperbolic_memory, dim).to(device)

print("\nBaseline Model (MemoryMLP):")
print(f"Parameters: {sum(p.numel() for p in baseline_model.parameters())}")

print("\nHyperbolic Model (HyperbolicMemoryMLP):")
# Count parameters carefully for hyperbolic model
total_params = 0
for p in hyperbolic_model.parameters():
    try:
        total_params += p.numel()
    except TypeError:
        # ManifoldParameter needs .tensor
        total_params += p.tensor.numel()
print(f"Parameters: {total_params}")

## 4. Training Functions

Define training and evaluation functions that:
1. Train the model on the memorization task
2. Track loss and metrics
3. Evaluate on validation data

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """
    Train for one epoch.
    
    Args:
        model: Model to train
        dataloader: Training data
        optimizer: Optimizer
        criterion: Loss function (MSE for regression)
        device: Device to train on
    
    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for x, y in tqdm(dataloader, desc="Training", leave=False):
        # Move data to device
        x, y = x.to(device), y.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        predictions = model(x)
        
        # Compute loss (Mean Squared Error between predicted and target vectors)
        loss = criterion(predictions, y)
        
        # Backward pass and optimize
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def evaluate(model, dataloader, criterion, device):
    """
    Evaluate model on validation/test data.
    
    Args:
        model: Model to evaluate
        dataloader: Validation/test data
        criterion: Loss function
        device: Device to evaluate on
    
    Returns:
        Dictionary with loss and cosine similarity metrics
    """
    model.eval()
    total_loss = 0.0
    total_cosine_sim = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            
            # Forward pass
            predictions = model(x)
            
            # Compute loss
            loss = criterion(predictions, y)
            total_loss += loss.item()
            
            # Compute cosine similarity (how aligned predictions are with targets)
            # Flatten to (batch * seq_len, dim)
            pred_flat = predictions.reshape(-1, predictions.size(-1))
            target_flat = y.reshape(-1, y.size(-1))
            
            # Normalize vectors
            pred_norm = torch.nn.functional.normalize(pred_flat, dim=-1)
            target_norm = torch.nn.functional.normalize(target_flat, dim=-1)
            
            # Compute cosine similarity
            cosine_sim = (pred_norm * target_norm).sum(dim=-1).mean()
            total_cosine_sim += cosine_sim.item()
            
            num_batches += 1
    
    return {
        'loss': total_loss / num_batches,
        'cosine_similarity': total_cosine_sim / num_batches
    }


def train_model(model, train_loader, val_loader, num_epochs=50, lr=1e-3):
    """
    Complete training loop.
    
    Args:
        model: Model to train
        train_loader: Training dataloader
        val_loader: Validation dataloader
        num_epochs: Number of epochs to train
        lr: Learning rate
    
    Returns:
        Dictionary with training history
    """
    # Loss function: Mean Squared Error for vector prediction
    criterion = nn.MSELoss()
    
    # Optimizer: AdamW with weight decay for regularization
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Learning rate scheduler: reduce on plateau
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    # Track history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_cosine_sim': [],
        'epoch_times': []
    }
    
    best_val_loss = float('inf')
    
    # Training loop
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Train for one epoch
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Evaluate on validation set
        val_metrics = evaluate(model, val_loader, criterion, device)
        val_loss = val_metrics['loss']
        val_cosine_sim = val_metrics['cosine_similarity']
        
        # Update learning rate based on validation loss
        scheduler.step(val_loss)
        
        # Record metrics
        epoch_time = time.time() - start_time
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_cosine_sim'].append(val_cosine_sim)
        history['epoch_times'].append(epoch_time)
        
        # Print progress
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}:")
            print(f"  Train Loss: {train_loss:.6f}")
            print(f"  Val Loss: {val_loss:.6f}")
            print(f"  Val Cosine Sim: {val_cosine_sim:.4f}")
            print(f"  Time: {epoch_time:.2f}s")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # In production, you'd save the model here
    
    return history

## 5. Train Baseline Model

Train the standard Euclidean MemoryMLP.

In [ ]:
print("="*60)
print("Training Baseline Model (MemoryMLP)")
print("="*60)

baseline_history = train_model(
    baseline_model,
    train_loader,
    val_loader,
    num_epochs=50,
    lr=1e-3
)

print("\n✓ Baseline training complete!")

## 6. Train Hyperbolic Model

Train the HyperbolicMemoryMLP that operates in Poincaré ball space.

In [ ]:
print("="*60)
print("Training Hyperbolic Model (HyperbolicMemoryMLP)")
print("="*60)

hyperbolic_history = train_model(
    hyperbolic_model,
    train_loader,
    val_loader,
    num_epochs=50,
    lr=1e-3
)

print("\n✓ Hyperbolic training complete!")

## 7. Compare Results

Visualize and compare the performance of both models.

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Training Loss
ax = axes[0, 0]
ax.plot(baseline_history['train_loss'], label='Baseline (MemoryMLP)', linewidth=2)
ax.plot(hyperbolic_history['train_loss'], label='Hyperbolic (HyperbolicMemoryMLP)', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss (MSE)', fontsize=12)
ax.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Validation Loss
ax = axes[0, 1]
ax.plot(baseline_history['val_loss'], label='Baseline (MemoryMLP)', linewidth=2)
ax.plot(hyperbolic_history['val_loss'], label='Hyperbolic (HyperbolicMemoryMLP)', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Loss (MSE)', fontsize=12)
ax.set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Cosine Similarity
ax = axes[1, 0]
ax.plot(baseline_history['val_cosine_sim'], label='Baseline (MemoryMLP)', linewidth=2)
ax.plot(hyperbolic_history['val_cosine_sim'], label='Hyperbolic (HyperbolicMemoryMLP)', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Prediction Alignment (Higher is Better)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Training Time per Epoch
ax = axes[1, 1]
ax.plot(baseline_history['epoch_times'], label='Baseline (MemoryMLP)', linewidth=2)
ax.plot(hyperbolic_history['epoch_times'], label='Hyperbolic (HyperbolicMemoryMLP)', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Training Time per Epoch', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plots saved to 'model_comparison.png'")

## 8. Numerical Comparison

Print detailed statistics comparing both models.

In [ ]:
print("="*70)
print("FINAL COMPARISON SUMMARY")
print("="*70)

# Get final metrics (average of last 5 epochs for stability)
baseline_final_train = np.mean(baseline_history['train_loss'][-5:])
baseline_final_val = np.mean(baseline_history['val_loss'][-5:])
baseline_final_cosine = np.mean(baseline_history['val_cosine_sim'][-5:])
baseline_avg_time = np.mean(baseline_history['epoch_times'])

hyperbolic_final_train = np.mean(hyperbolic_history['train_loss'][-5:])
hyperbolic_final_val = np.mean(hyperbolic_history['val_loss'][-5:])
hyperbolic_final_cosine = np.mean(hyperbolic_history['val_cosine_sim'][-5:])
hyperbolic_avg_time = np.mean(hyperbolic_history['epoch_times'])

print("\n📊 BASELINE MODEL (MemoryMLP)")
print("-" * 70)
print(f"Final Training Loss:      {baseline_final_train:.6f}")
print(f"Final Validation Loss:    {baseline_final_val:.6f}")
print(f"Final Cosine Similarity:  {baseline_final_cosine:.4f}")
print(f"Avg Training Time/Epoch:  {baseline_avg_time:.2f}s")

print("\n🌀 HYPERBOLIC MODEL (HyperbolicMemoryMLP)")
print("-" * 70)
print(f"Final Training Loss:      {hyperbolic_final_train:.6f}")
print(f"Final Validation Loss:    {hyperbolic_final_val:.6f}")
print(f"Final Cosine Similarity:  {hyperbolic_final_cosine:.4f}")
print(f"Avg Training Time/Epoch:  {hyperbolic_avg_time:.2f}s")

print("\n📈 RELATIVE IMPROVEMENTS")
print("-" * 70)

# Compute relative differences
val_loss_improvement = ((baseline_final_val - hyperbolic_final_val) / baseline_final_val) * 100
cosine_improvement = ((hyperbolic_final_cosine - baseline_final_cosine) / baseline_final_cosine) * 100
time_overhead = ((hyperbolic_avg_time - baseline_avg_time) / baseline_avg_time) * 100

print(f"Validation Loss Change:   {val_loss_improvement:+.2f}% {'(better)' if val_loss_improvement > 0 else '(worse)'}")
print(f"Cosine Similarity Change: {cosine_improvement:+.2f}% {'(better)' if cosine_improvement > 0 else '(worse)'}")
print(f"Training Time Overhead:   {time_overhead:+.2f}%")

print("\n🏆 WINNER")
print("-" * 70)

# Determine winner based on validation loss
if hyperbolic_final_val < baseline_final_val:
    winner = "Hyperbolic Model"
    improvement = val_loss_improvement
elif baseline_final_val < hyperbolic_final_val:
    winner = "Baseline Model"
    improvement = -val_loss_improvement
else:
    winner = "Tie"
    improvement = 0

if winner != "Tie":
    print(f"{winner} wins with {improvement:.2f}% better validation loss!")
else:
    print("Both models perform equally well!")

print("\n" + "="*70)

## 9. Qualitative Analysis

Analyze a sample prediction to visualize how each model performs.

In [ ]:
# Get a sample from validation set
sample_x, sample_y = next(iter(val_loader))
sample_x = sample_x[:1].to(device)  # Take first sequence
sample_y = sample_y[:1].to(device)

# Get predictions from both models
baseline_model.eval()
hyperbolic_model.eval()

with torch.no_grad():
    baseline_pred = baseline_model(sample_x)
    hyperbolic_pred = hyperbolic_model(sample_x)

# Move to CPU for visualization
sample_y = sample_y.cpu().squeeze(0)
baseline_pred = baseline_pred.cpu().squeeze(0)
hyperbolic_pred = hyperbolic_pred.cpu().squeeze(0)

# Compute per-position errors
baseline_errors = torch.norm(baseline_pred - sample_y, dim=-1).numpy()
hyperbolic_errors = torch.norm(hyperbolic_pred - sample_y, dim=-1).numpy()

# Plot prediction errors over sequence
plt.figure(figsize=(12, 5))

plt.plot(baseline_errors, 'o-', label='Baseline Error', linewidth=2, markersize=6)
plt.plot(hyperbolic_errors, 's-', label='Hyperbolic Error', linewidth=2, markersize=6)

plt.xlabel('Sequence Position', fontsize=12)
plt.ylabel('L2 Prediction Error', fontsize=12)
plt.title('Per-Position Prediction Error on Sample Sequence', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('prediction_errors.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage error on this sample:")
print(f"  Baseline:   {baseline_errors.mean():.4f}")
print(f"  Hyperbolic: {hyperbolic_errors.mean():.4f}")
print("\n✓ Prediction error plot saved to 'prediction_errors.png'")

## 10. Key Insights

### What did we learn?

1. **Memory Capacity**: Both models can memorize and predict sequences, but their effectiveness may differ

2. **Hyperbolic Geometry**: The HyperbolicMemoryMLP operates in Poincaré ball space, which:
   - Can represent hierarchical structures more efficiently
   - May be beneficial for data with tree-like relationships
   - Has curved geometry that might capture certain patterns better

3. **Trade-offs**:
   - **Baseline Model**: Simpler, faster, well-understood
   - **Hyperbolic Model**: More expressive geometry, potential for hierarchical data

4. **Computational Cost**: The hyperbolic model adds overhead due to exponential/logarithmic maps

### When to use which?

- **Use Baseline (MemoryMLP)** when:
  - Data has no clear hierarchical structure
  - Training time is critical
  - Simplicity and interpretability are priorities

- **Use Hyperbolic (HyperbolicMemoryMLP)** when:
  - Data has hierarchical or tree-like structure
  - Working with graphs, taxonomies, or nested relationships
  - Extra computational cost is acceptable for potential performance gains

## 11. Save Results

Save the training histories for future analysis.

In [ ]:
import pickle

# Save training histories
results = {
    'baseline_history': baseline_history,
    'hyperbolic_history': hyperbolic_history,
    'config': {
        'dim': dim,
        'seq_length': seq_length,
        'num_train': len(train_dataset),
        'num_val': len(val_dataset),
        'num_epochs': 50
    }
}

with open('training_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("✓ Results saved to 'training_results.pkl'")
print("\n" + "="*70)
print("EXPERIMENT COMPLETE!")
print("="*70)